# Trade Reconciliation Tool

Compares an internal trade blotter with a broker statement, finds every mismatch ("break"),
labels it and values it in USD, then produces an Excel report and a chart.

**Input:** two CSV files (internal records + broker statement)
**Output:** `recon_report_<date>.xlsx` (Summary, Breaks, Matched tabs) + `breaks_by_type.png`

**How to run:** set the file paths in the "Run it" cell and run all cells.

## 1. Settings
All settings live here, so nothing below needs editing.
Each data source has a config describing its column names, date format and buy/sell codes.
To add a new broker, copy `BROKER_CONFIG` and change the values.

In [1]:
import os
from datetime import date
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_FOLDER = "Output/"

PRICE_TOLERANCE = {"EURUSD": 0.0001, "GBPUSD": 0.0001, "XAUUSD": 0.05}

COLUMNS = ["trade_id", "trade_date", "settle_date", "symbol",
           "side", "quantity", "price", "counterparty"]

INTERNAL_CONFIG = {
    "column_map":  {},
    "date_format": "%Y-%m-%d",
    "side_map":    {},
}

BROKER_CONFIG = {
    "column_map": {"TradeRef": "trade_id", "Trade Date": "trade_date",
                   "Instrument": "symbol", "B/S": "side", "Qty": "quantity",
                   "Exec Price": "price", "Cpty": "counterparty",
                   "Value Date": "settle_date"},
    "date_format": "%d/%m/%Y",
    "side_map":    {"B": "BUY", "S": "SELL"},
}

## 2. Load and clean
Reads a trade file and converts it to one standard layout: same column names, clean text,
real dates and numbers. Duplicates are flagged, not deleted, because a duplicate is a break.
Stops with a clear message if a file is missing columns or contains blanks.

In [2]:
def load_and_clean(path, config):
    """Read one trade file and return it in the standard layout."""
    df = pd.read_csv(path).rename(columns=config["column_map"])

    missing = set(COLUMNS) - set(df.columns)
    assert not missing, f"{path} is missing columns: {missing}"

    for col in ["trade_id", "symbol", "side", "counterparty"]:
        df[col] = df[col].astype(str).str.strip()
    for col in ["trade_id", "symbol", "side"]:
        df[col] = df[col].str.upper()
    df["side"] = df["side"].replace(config["side_map"])

    for col in ["trade_date", "settle_date"]:
        df[col] = pd.to_datetime(df[col], format=config["date_format"])

    df["quantity"] = pd.to_numeric(df["quantity"]).astype("int64")
    df["price"]    = pd.to_numeric(df["price"]).astype("float64")

    df = df[COLUMNS].copy()
    df["is_duplicate"] = df.duplicated(subset="trade_id", keep="first")

    assert df[COLUMNS].isna().sum().sum() == 0, f"{path} has blank cells"
    assert set(df["side"]) <= {"BUY", "SELL"}, f"{path} has unknown side codes"
    return df

## 3. Match trades
Joins the two files on trade ID, keeping trades found on only one side.
Matched trades are compared field by field. Price allows a small tolerance for rounding;
quantity, side and dates must match exactly. Every trade gets a break type.

In [3]:
CHECK_LABELS = {"price_ok": "Price", "qty_ok": "Quantity", "side_ok": "Side",
                "settle_ok": "Settle Date", "tdate_ok": "Trade Date",
                "symbol_ok": "Symbol"}

def match_trades(internal, broker, tolerance=PRICE_TOLERANCE):
    """Match two clean tables and return one row per trade with a break type."""
    # --- Phase 4: set aside duplicates and join ---
    dupes = broker[broker["is_duplicate"]]
    merged = internal[~internal["is_duplicate"]].drop(columns="is_duplicate").merge(
        broker[~broker["is_duplicate"]].drop(columns="is_duplicate"),
        on="trade_id", how="outer", suffixes=("_int", "_brk"), indicator=True)

    # --- Phase 4: compare the fields ---
    both = merged[merged["_merge"] == "both"].copy()
    both["price_ok"]  = (both["price_brk"] - both["price_int"]).abs() <= both["symbol_int"].map(tolerance)
    both["qty_ok"]    = both["quantity_int"]    == both["quantity_brk"]
    both["side_ok"]   = both["side_int"]        == both["side_brk"]
    both["settle_ok"] = both["settle_date_int"] == both["settle_date_brk"]
    both["tdate_ok"]  = both["trade_date_int"]  == both["trade_date_brk"]
    both["symbol_ok"] = both["symbol_int"]      == both["symbol_brk"]

    # --- Phase 5: label each trade ---
    def label(row):
        failed = [name for check, name in CHECK_LABELS.items() if not row[check]]
        return " + ".join(failed) if failed else "Matched"
    both["break_type"] = both.apply(label, axis=1) if len(both) else pd.Series(dtype=str)

    missing_brk = merged[merged["_merge"] == "left_only"].assign(break_type="Missing in Broker")
    missing_int = merged[merged["_merge"] == "right_only"].assign(break_type="Missing Internal")
    dupes = (dupes.drop(columns="is_duplicate").add_suffix("_brk")
             .rename(columns={"trade_id_brk": "trade_id"}).assign(break_type="Duplicate"))

    # --- Phase 5: one master table ---
    recon = pd.concat([both, missing_brk, missing_int, dupes], ignore_index=True)

    for col in COLUMNS[1:]:
        recon[col] = recon[col + "_int"].fillna(recon[col + "_brk"])
    recon["quantity"] = recon["quantity"].astype("int64")
    recon["notional_usd"] = (recon["quantity"] * recon["price"]).round(2)
    recon["status"] = (recon["break_type"] == "Matched").map({True: "Matched", False: "Break"})

    master_cols = ["trade_id", "status", "break_type"] + COLUMNS[1:] + ["notional_usd"]
    return (recon[master_cols]
            .sort_values(["status", "break_type", "trade_id"])
            .reset_index(drop=True))

## 4. Summary
Counts the trades and totals the USD notional for each break type, with a TOTAL row.

In [4]:
def build_summary(master):
    """Count and value of trades per break type, with a TOTAL row."""
    summary = (master.groupby("break_type")
               .agg(trades=("trade_id", "count"),
                    notional_usd=("notional_usd", "sum"))
               .sort_values("notional_usd", ascending=False)
               .reset_index())
    total = pd.DataFrame([{"break_type": "TOTAL",
                           "trades": summary["trades"].sum(),
                           "notional_usd": summary["notional_usd"].sum()}])
    return pd.concat([summary, total], ignore_index=True)

## 5. Chart
Shows which break types carry the most money, so the team knows what to chase first.

In [5]:
def plot_breaks(summary, folder=OUTPUT_FOLDER):
    """Bar chart of break value by type; saves a PNG and returns its path."""
    chart = (summary[~summary["break_type"].isin(["Matched", "TOTAL"])]
             .sort_values("notional_usd"))

    fig, ax = plt.subplots(figsize=(8, 4))

    if chart.empty:
        ax.text(0.5, 0.5, "No breaks found", ha="center", va="center", fontsize=14)
        ax.axis("off")
    else:
        bars = ax.barh(chart["break_type"], chart["notional_usd"] / 1e6,
                       color="#2a78d6", height=0.6)
        for bar, n in zip(bars, chart["trades"]):
            ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
                    f"${bar.get_width():.1f}M  ({n} trades)",
                    va="center", fontsize=9, color="#52514e")
        ax.set_title("Notional value of breaks by type (USD millions)", loc="left", fontsize=12)
        ax.set_xlabel("USD millions", color="#52514e")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="x", color="#e6e6e2", linewidth=0.8)
        ax.set_axisbelow(True)
        ax.set_xlim(0, chart["notional_usd"].max() / 1e6 * 1.35)

    plt.tight_layout()
    path = folder + "breaks_by_type.png"
    plt.savefig(path, dpi=150)
    plt.show()
    return path

## 6. Excel report
Saves a dated Excel file with three tabs: Summary for managers, Breaks for the team
investigating, and Matched as the audit trail.

In [6]:
def save_report(master, summary, folder=OUTPUT_FOLDER):
    """Write Summary / Breaks / Matched tabs to a dated Excel file."""
    path = folder + f"recon_report_{date.today():%Y-%m-%d}.xlsx"

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="Summary", index=False)
        master[master["status"] == "Break"].to_excel(writer, sheet_name="Breaks", index=False)
        master[master["status"] == "Matched"].to_excel(writer, sheet_name="Matched", index=False)

        for sheet in writer.sheets.values():
            for col in sheet.columns:
                width = max(len(str(c.value)) if c.value is not None else 0 for c in col)
                sheet.column_dimensions[col[0].column_letter].width = width + 3
            sheet.freeze_panes = "A2"

    return path

## 7. Run the full reconciliation
Connects every step: clean both files, match, summarise, save the report and draw the chart.

In [7]:
def run_recon(internal_path, broker_path,
              internal_config=INTERNAL_CONFIG, broker_config=BROKER_CONFIG,
              folder=OUTPUT_FOLDER):
    """Run the whole reconciliation: two CSVs in, Excel report + chart out."""
    os.makedirs(folder, exist_ok=True)

    internal = load_and_clean(internal_path, internal_config)
    broker   = load_and_clean(broker_path, broker_config)
    master   = match_trades(internal, broker)
    summary  = build_summary(master)
    report   = save_report(master, summary, folder)
    chart    = plot_breaks(summary, folder)

    breaks = master[master["status"] == "Break"]
    print(f"Trades checked: {len(master)}")
    print(f"Match rate:     {(master['status'] == 'Matched').mean():.1%}")
    print(f"Breaks:         {len(breaks)}  (${breaks['notional_usd'].sum():,.0f} at risk)")
    print(f"Report:         {report}")
    print(f"Chart:          {chart}")

    return master, summary

## 8. Run it
Change the two file paths to reconcile a different pair of files.

In [ ]:
master, summary = run_recon("Raw Data/internal_blotter.csv",
                            "Raw Data/broker_statement.csv")
display(summary)

## 9. Tests
Proves the tool works:
- **Answer key:** all 23 planted breaks are caught with the correct label, and the 3
  small price differences inside tolerance are matched.
- **Sanity check:** comparing a file with itself gives zero breaks.

In [ ]:
key = pd.read_csv("Raw Data/answer_key.csv")
should_match = key[key["break_type"].str.startswith("Within")]
key = key[~key["break_type"].str.startswith("Within")]

found = master.loc[master["status"] == "Break", ["trade_id", "break_type"]]
check = key.merge(found, on="trade_id", how="outer", suffixes=("_expected", "_found"))

assert (check["break_type_expected"] == check["break_type_found"]).all(), check
assert (master.loc[master["trade_id"].isin(should_match["trade_id"]), "status"] == "Matched").all()
print("✅ All 23 planted breaks caught, all 3 tolerance trades matched")

# Sanity test: the same file on both sides must give zero breaks
m2, _ = run_recon("Raw Data/internal_blotter.csv", "Raw Data/internal_blotter.csv",
                  broker_config=INTERNAL_CONFIG, folder="Output/sanity_test/")
assert (m2["status"] == "Matched").all()
print("✅ Same file on both sides gives zero breaks")